1. Загрузить embedding.
2. Исследовать формат bytes.
3. Понять, откуда берётся размерность 312.
4. Корректно декодировать один embedding.
5. Проверить диапазон значений.
6. Проверить нормы векторов.
7. Проверить cosine similarity на нескольких товарах.
8. Только после этого переходить к массовому декодированию выборки.

In [10]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import struct

DATA_PATH = "../data/raw/dataset-electronics-5M.parquet"

parquet_file = pq.ParquetFile(DATA_PATH)

row_group = parquet_file.read_row_group(
    0,
    columns=["embedding", "model_text", "category_id", "model_id"]
).to_pandas()

print("Размер row group:", row_group.shape)

Размер row group: (3578, 4)


In [1]:
import os

print(os.path.exists("../data/embeddings_sample.npy"))
print(os.path.abspath("../data/embeddings_sample.npy"))

True
/Users/denis/Desktop/coursework/Course_work_community_detection/data/embeddings_sample.npy


In [4]:
embeddings = np.load("../data/embeddings_sample.npy")

print("Размер:", embeddings.shape)
print("Тип:", embeddings.dtype)
print("Минимум:", embeddings.min())
print("Максимум:", embeddings.max())
print("Среднее:", embeddings.mean())
print("Стандартное отклонение:", embeddings.std())

Размер: (1000, 312)
Тип: float32
Минимум: -5.0598245
Максимум: 3.2596614
Среднее: 0.018396575
Стандартное отклонение: 0.6042406


* размерность (1000, 312) — значит мы действительно получили 312-мерные векторы;
* float32 — подходящий тип;
* среднее 0.018 близко к нулю;
* стандартное отклонение 0.604 — значения не выглядят как мусор;
* диапазон [-5.06; 3.26] тоже сам по себе нормален для ненормализованных эмбеддингов.

Следующий шаг — проверим нормы векторов.

In [5]:
embedding_norms = np.linalg.norm(
    embeddings,
    axis=1
)

print("Минимальная норма:", embedding_norms.min())
print("Максимальная норма:", embedding_norms.max())
print("Средняя норма:", embedding_norms.mean())
print("Медианная норма:", np.median(embedding_norms))

Минимальная норма: 8.777028
Максимальная норма: 12.334148
Средняя норма: 10.6661
Медианная норма: 10.719985


In [6]:
print(
    "Векторов с нормой < 1e-6:",
    np.sum(embedding_norms < 1e-6)
)

Векторов с нормой < 1e-6: 0


In [7]:
import os

data_dir = "../data"

print(os.listdir(data_dir))

['embeddings', 'raw', 'embeddings_sample.npy']


In [8]:
import os

print(os.listdir("../data/embeddings"))

[]


In [13]:
embedding_bytes = row_group.iloc[0]["embedding"]

print("Размер bytes:", len(embedding_bytes))
print("312 float32:", 312 * 4)
print("Разница:", len(embedding_bytes) - 312 * 4)

Размер bytes: 3122
312 float32: 1248
Разница: 1874


In [ ]:
import os

for root, dirs, files in os.walk(".."):
    for file in files:
        if file.endswith(".ipynb") or file.endswith(".py")

../check_model.py
../.venv/lib/python3.13/site-packages/nest_asyncio2.py
../.venv/lib/python3.13/site-packages/threadpoolctl.py
../.venv/lib/python3.13/site-packages/ipython_pygments_lexers.py
../.venv/lib/python3.13/site-packages/pylab.py
../.venv/lib/python3.13/site-packages/jsonpointer.py
../.venv/lib/python3.13/site-packages/ipykernel_launcher.py
../.venv/lib/python3.13/site-packages/pandocfilters.py
../.venv/lib/python3.13/site-packages/isympy.py
../.venv/lib/python3.13/site-packages/six.py
../.venv/lib/python3.13/site-packages/rfc3339_validator.py
../.venv/lib/python3.13/site-packages/rfc3986_validator.py
../.venv/lib/python3.13/site-packages/typing_extensions.py
../.venv/lib/python3.13/site-packages/jupyter.py
../.venv/lib/python3.13/site-packages/fastjsonschema/draft06.py
../.venv/lib/python3.13/site-packages/fastjsonschema/version.py
../.venv/lib/python3.13/site-packages/fastjsonschema/draft07.py
../.venv/lib/python3.13/site-packages/fastjsonschema/draft2019.py
../.venv/lib/py

In [16]:
embedding = df["embedding"].iloc[0]

print("Тип:", type(embedding))
print("Размер:", len(embedding))

print("\nHEX:")
print(embedding.hex())

print("\nПервые 128 байт:")
print(embedding[:128])

print("\nПоследние 128 байт:")
print(embedding[-128:])

Тип: <class 'bytes'>
Размер: 3122

HEX:
5b03000000000000b5bf3b0300000000006076bf3b03000000000040b4bf3b030000000000a0973f3b030000000000c07d3f3b030000000000a0c3bf3b0300000000006096bf3b030000000000e083bf3b03000000000060813f3b03000000000040a5bf3b030000000000a0503f3b03000000000060bebf3b030000000000009dbf3b03000000000060913f3b030000000000c0803f3b03000000000060843f3b03000000000020b73f3b030000000000a08ebf3b03000000000000963f3b030000000000406abf3b030000000000809f3f3b030000000000c0773f3b03000000000020a93f3b030000000000a0a03f3b03000000000060813f3b03000000000020b2bf3b0300000000004061bf3b030000000000a0963f3b03000000000020803f3b03000000000040ac3f3b03000000000020a2bf3b030000000000c0abbf3b030000000000e09cbf3b03000000000060bb3f3b0300000000002073bf3b030000000000809cbf3b03000000000040aabf3b03000000000040a8bf3b03000000000060acbf3b030000000000a0acbf3b03000000000080b43f3b03000000000020703f3b03000000000080babf3b030000000000005b3f3b030000000000e0883f3b030000000000a0733f3b03000000000080a63f3b03000000000040ba3f

In [17]:
for i in range(5):
    emb = df["embedding"].iloc[i]
    print(f"Embedding {i}: {len(emb)} bytes")

Embedding 0: 3122 bytes
Embedding 1: 3122 bytes
Embedding 2: 3122 bytes
Embedding 3: 3122 bytes
Embedding 4: 3122 bytes


In [18]:
embedding_sizes = df["embedding"].apply(len)

print(embedding_sizes.value_counts())
print("Уникальных размеров:", embedding_sizes.nunique())

embedding
3122    3578
Name: count, dtype: int64
Уникальных размеров: 1


In [21]:
from pathlib import Path

MODEL_PATH = Path("../models/epoch_64_encoder.pth")

print("Файл существует:", MODEL_PATH.exists())

if MODEL_PATH.exists():
    print("Размер:", MODEL_PATH.stat().st_size / 1024**3, "GB")

Файл существует: True
Размер: 0.325231047347188 GB


In [22]:
DATA_PATH = Path("../data/raw/dataset-electronics-5M.parquet")

print("Parquet существует:", DATA_PATH.exists())

if DATA_PATH.exists():
    print("Размер:", DATA_PATH.stat().st_size / 1024**3, "GB")

Parquet существует: True
Размер: 7.077627279795706 GB


In [23]:
import io
import torch

embedding_bytes = df["embedding"].iloc[0]

try:
    tensor = torch.load(
        io.BytesIO(embedding_bytes),
        map_location="cpu"
    )

    print("Успешно!")
    print("Тип:", type(tensor))

    if hasattr(tensor, "shape"):
        print("Shape:", tensor.shape)

    if hasattr(tensor, "dtype"):
        print("Dtype:", tensor.dtype)

except Exception as e:
    print("Не удалось загрузить как torch:", type(e).__name__)
    print(e)

AttributeError: partially initialized module 'torch' from '/Users/denis/Desktop/coursework/Course_work_community_detection/.venv/lib/python3.13/site-packages/torch/__init__.py' has no attribute '_logging' (most likely due to a circular import)

In [24]:
import sys
from pathlib import Path

print("Python:")
print(sys.executable)

print("\nТекущая директория:")
print(Path.cwd())

print("\nФайлы/папки с названием torch:")
for path in Path(".").rglob("torch*"):
    print(path)

Python:
/Users/denis/Desktop/coursework/Course_work_community_detection/.venv/bin/python

Текущая директория:
/Users/denis/Desktop/coursework/Course_work_community_detection/notebooks

Файлы/папки с названием torch:


In [27]:
import torch

print("PyTorch:", torch.__version__)
print("MPS:", torch.backends.mps.is_available())

AttributeError: partially initialized module 'torch' from '/Users/denis/Desktop/coursework/Course_work_community_detection/.venv/lib/python3.13/site-packages/torch/__init__.py' has no attribute 'fx' (most likely due to a circular import)

In [33]:
import sys
print(sys.executable)

/Users/denis/Desktop/coursework/Course_work_community_detection/.venv/bin/python


In [ ]:
import torch

print(torch.__version__)
print(torch.__file__)

AttributeError: partially initialized module 'torch' from '/Users/denis/Desktop/coursework/Course_work_community_detection/.venv/lib/python3.13/site-packages/torch/__init__.py' has no attribute 'fx' (most likely due to a circular import)

: 